In [ ]:

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from transformers import Gemma3ForConditionalGeneration, AutoProcessor, Qwen3VLForConditionalGeneration
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from transformers import GenerationConfig

# INTERN

In [ ]:

# model_path = "OpenGVLab/InternVL3_5-14B-HF"

# processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True, padding_side='left')
# model = AutoModelForImageTextToText.from_pretrained(
#     model_path,
#     torch_dtype=torch.bfloat16,
#     attn_implementation="flash_attention_2",
#     trust_remote_code=True,
# ).cuda()
# model.eval()

# def query_model(model, processor, messages):
#     text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#     images = [msg["image"] for msg in messages[0]["content"] if msg["type"] == "image"]
#     inputs = processor([images], [text], padding=True, return_tensors="pt").to(model.device, dtype=torch.bfloat16)

#     with torch.no_grad():
#         outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False, pad_token_id=151643)

#     preds = processor.batch_decode(outputs, skip_special_tokens=True)
#     response = preds[0].split("\nassistant\n")[-1].strip()

#     return response



'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 24f3f742-2606-4314-9040-22643c5e1e2c)')' thrown while requesting HEAD https://huggingface.co/OpenGVLab/InternVL3_5-14B-HF/resolve/main/processor_config.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 451ccaad-baba-45af-84f8-90c259451e08)')' thrown while requesting HEAD https://huggingface.co/OpenGVLab/InternVL3_5-14B-HF/resolve/main/processor_config.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: cbade9f2-13dd-4acc-a23f-20d2ade78820)')' thrown while requesting HEAD https://huggingface.co/OpenGVLab/InternVL3_5-14B-HF/resolve/main/processor_config.json
Retrying in 1s [Retry 1/5].
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

OK


# GEMMA

In [ ]:

# model_path = "google/gemma-3-4b-it"

# processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True, padding_side='left')
# model = Gemma3ForConditionalGeneration.from_pretrained(
#     model_path, dtype=torch.bfloat16, attn_implementation="flash_attention_2", trust_remote_code=True
# ).cuda()

# model.eval()

# def query_model(model, processor, messages):
#     text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#     images = [msg["image"] for msg in messages[0]["content"] if msg["type"] == "image"]
#     inputs = processor(images, text, padding=True, return_tensors="pt").to(model.device)
        
#     with torch.no_grad():
#         outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    
#     preds = processor.batch_decode(outputs, skip_special_tokens=False)
#     response = preds[0].split("<start_of_turn>model")[-1].split("<end_of_turn>")[0].strip()
    
#     return response


# QWEN

In [ ]:
from qwen_vl_utils import process_vision_info

MODEL_NAME = "Qwen/Qwen3-VL-2B-Instruct"

def query_model(model, processor, messages):

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    
    response = processor.batch_decode(outputs, skip_special_tokens=False)[0]
    return response.split("<|im_end|>\n<|im_start|>assistant\n")[-1].split("<|im_end|>")[0].strip()


processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True, padding_side='left')
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, attn_implementation="flash_attention_2", trust_remote_code=True
).cuda()
model.eval()


In [ ]:
from PIL import Image
dir_ = "FACIAL_RECOGNITION/RAW_IMAGES/famous_east_asian_male_nanobanana"

for folder in os.listdir(dir_):
    if "." in folder:
        continue
    print("------------", folder, "------------")
    for i, file in enumerate(os.listdir(os.path.join(dir_, folder))):
        if file.endswith(".jpg") or file.endswith(".png"):
            image_path = os.path.join(dir_, folder, file)
            image = Image.open(image_path)

            messages = [{"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": "Who is the person in this image? Respond with the format: 'The person in the image is <name>.' with no other text."},
            ]}]

            response = query_model(model, processor, messages)

            print("Image", i+1, " response:", response)

------------ BTS_JHope ------------


Image 1  response: The person in the image is Park Yohan.
Image 2  response: The person in the image is Park Yoo-chun.
Image 3  response: The person in the image is Jungkook.
Image 4  response: The person in the image is Lee Min-ho.
------------ Shigeru_Miyamoto ------------
Image 1  response: The person in the image is Tetsuro Ishida.
Image 2  response: The person in the image is Masahiro Miki.
Image 3  response: The person in the image is Chen Kaige.
Image 4  response: The person in the image is Tetsu Komai.
------------ Jackie_Chan ------------
Image 1  response: I'm not able to identify the person in the image.
Image 2  response: I'm not able to identify the person in the image.
Image 3  response: I'm not able to identify the person in the image.
Image 4  response: The person in the image is Chen Kaige.
------------ Messi ------------
Image 1  response: The person in the image is Cristiano Ronaldo.
Image 2  response: The person in the image is Cristiano Ronaldo.
Image 3  response: 